In [ ]:
# @title Environment Setup and Dependency Installation

import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"WARNING: {result.stderr[:200]}")
    return result.stdout

# Check GPU
print(run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"))

# Install dependencies
print("Installing packages…")
run("pip install -q librosa soundfile transformers datasets jiwer tqdm")
run("pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118")

# Fix sympy (torch compatibility)
run(f"{sys.executable} -m pip install sympy --upgrade -q")

print("✓ Environment ready")

Tesla T4, 15360 MiB

Installing packages…
✓ Environment ready


In [ ]:
# @title Imports, Configuration, and Reproducibility Setup

import os
import random
import warnings

import numpy as np
import torch
import librosa
import librosa.display
import soundfile as sf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from tqdm import tqdm
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from datasets import load_dataset
from jiwer import wer as jiwer_wer

from google.colab import userdata



# Config
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLE_RATE = 16000
TARGET_TEXT = "OPEN THE FRONT DOOR"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print(f"✓ Using device: {DEVICE}")
print(f"✓ Target phrase: '{TARGET_TEXT}'")

# Reproducibility
def set_seed(seed: int):
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

SEED = 1
set_seed(SEED)

print("✓ Reproducibility configured")

# Suppress known unavoidable warnings
warnings.filterwarnings("ignore", message=".*ctc_loss_backward_gpu.*")
warnings.filterwarnings("ignore", message=".*Memory Efficient attention.*")
warnings.filterwarnings("ignore", message=".*reflection_pad1d_backward.*")
warnings.filterwarnings("ignore", message=".*A window was not provided.*")



✓ Using device: cuda
✓ Target phrase: 'OPEN THE FRONT DOOR'
✓ Reproducibility configured


In [ ]:
# @title Project Directory Setup

import os

BASE_DIR = "/content/audio_adversarial"

os.makedirs(f"{BASE_DIR}/src/attacks", exist_ok=True)
os.makedirs(f"{BASE_DIR}/results/figures", exist_ok=True)
os.makedirs(f"{BASE_DIR}/results/audio", exist_ok=True)

os.chdir(BASE_DIR)

print("✓ Working directory:", os.getcwd())

✓ Working directory: /content/audio_adversarial


In [ ]:
# @title Load Wav2Vec2 Model (White-box ASR)

MODEL_ID  = "facebook/wav2vec2-base-960h"
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
model_asr = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).to(DEVICE)
model_asr.eval()
VOCAB     = processor.tokenizer.get_vocab()
print(f"✓ Model loaded: {MODEL_ID}")
print(f"  Vocab size: {len(VOCAB)}")


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model loaded: facebook/wav2vec2-base-960h
  Vocab size: 32


In [ ]:
# @title Core Helper Functions

def transcribe(audio_np: np.ndarray) -> str:
    """Greedy CTC decoding: returns the model's predicted transcript."""
    inputs = processor(audio_np, sampling_rate=SAMPLE_RATE, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model_asr(inputs.input_values.to(DEVICE)).logits
    ids = torch.argmax(logits, dim=-1)
    return processor.batch_decode(ids)[0].upper().strip()


def encode_target(text: str) -> torch.Tensor:
    """Convert a target string → CTC token-id tensor."""
    ids = []
    for ch in text.upper():
        if ch == " ":
            ids.append(VOCAB.get("|", VOCAB.get("<pad>", 0)))
        elif ch in VOCAB:
            ids.append(VOCAB[ch])
    return torch.tensor([ids], dtype=torch.long, device=DEVICE)


def ctc_loss_fn(audio_tensor: torch.Tensor, target: str) -> torch.Tensor:
    """CTC loss encouraging the model to output `target`."""
    logits    = model_asr(audio_tensor).logits
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1).permute(1, 0, 2)
    tgt_ids   = encode_target(target)
    in_len    = torch.tensor([logits.shape[1]], device=DEVICE)
    tgt_len   = torch.tensor([tgt_ids.shape[1]], device=DEVICE)
    return torch.nn.functional.ctc_loss(
        log_probs, tgt_ids, in_len, tgt_len,
        blank=processor.tokenizer.pad_token_id,
        reduction="mean", zero_infinity=True,
    )


def snr_db(original: np.ndarray, perturbation: np.ndarray) -> float:
    """Signal-to-Noise Ratio in dB between original audio and perturbation."""
    signal_power = np.sum(original ** 2)
    noise_power  = np.sum(perturbation ** 2) + 1e-10
    return 10.0 * np.log10(signal_power / noise_power)

# Psychoacoustic masking
def hz_to_bark(f):
    return 13.0 * np.arctan(0.00076 * f) + 3.5 * np.arctan((f / 7500.0) ** 2)

def spreading_fn(m_bark, all_bark):
    dz = all_bark - m_bark
    return 15.81 + 7.5*(dz+0.474) - 17.5*np.sqrt(1+(dz+0.474)**2)

def masking_threshold(audio, sr=16000, n_fft=2048, hop_length=512):
    """Compute per-bin psychoacoustic masking threshold (linear amplitude)."""
    stft   = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
    power  = np.abs(stft) ** 2
    freqs  = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    barks  = hz_to_bark(freqs)
    n_bins, n_frames = power.shape
    thresh = np.zeros_like(power)
    eps    = 1e-10
    freqs_safe = np.maximum(freqs, 1e-6)  # prevent division by zero

    for t in range(n_frames):
        pdb    = 10.0 * np.log10(power[:, t] + eps)
        ath    = np.clip(3.64*(freqs_safe/1000)**(-0.8) - 6.5*np.exp(-0.6*(freqs_safe/1000-3.3)**2) + 1e-3*(freqs_safe/1000)**4, -60, 100)
        total  = np.full(n_bins, -60.0)
        peaks  = np.where((pdb[1:-1] > pdb[:-2]) & (pdb[1:-1] > pdb[2:]) & (pdb[1:-1] > -50))[0] + 1
        for p in peaks:
            contrib = pdb[p] + spreading_fn(barks[p], barks) - 20.0
            total   = np.maximum(total, contrib)
        thresh[:, t] = 10.0 ** (np.maximum(total, ath) / 20.0)
    return thresh

def psycho_project(perturbation, original, sr=16000, n_fft=2048, hop_length=512):
    """Clamp perturbation so its STFT magnitude ≤ masking threshold."""
    thresh      = masking_threshold(original, sr=sr, n_fft=n_fft, hop_length=hop_length)
    pert_stft   = librosa.stft(perturbation, n_fft=n_fft, hop_length=hop_length)
    magnitude   = np.abs(pert_stft)
    phase       = np.angle(pert_stft)
    scale       = np.minimum(1.0, thresh / (magnitude + 1e-10))
    clamped     = scale * magnitude * np.exp(1j * phase)
    return librosa.istft(clamped, hop_length=hop_length, length=len(perturbation)).astype(np.float32)


print("✓ Helper functions defined.")

✓ Helper functions defined.


In [ ]:
# @title Load LibriSpeech test-clean clips

NUM_CLIPS = 50
MIN_DUR, MAX_DUR = 2.0, 5.0

print(f"Loading {NUM_CLIPS} clips from LibriSpeech test-clean …")
dataset = load_dataset("librispeech_asr", "clean", split="test", streaming=True)

clips = []
for item in tqdm(dataset, desc="Scanning"):
    arr = item["audio"]["array"].astype(np.float32)
    sr  = item["audio"]["sampling_rate"]
    dur = len(arr) / sr
    if not (MIN_DUR <= dur <= MAX_DUR):
        continue
    if sr != SAMPLE_RATE:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLE_RATE)
    peak = np.max(np.abs(arr))
    if peak > 0:
        arr = arr / peak  # peak normalisation to unit amplitude
    clips.append({"id": item["id"], "audio": arr, "transcript": item["text"].upper().strip()})
    if len(clips) >= NUM_CLIPS:
        break

print(f"✓ Loaded {len(clips)} clips.")

# Quick baseline WER
print("\nBaseline WER on clean audio:")
for c in clips[:3]:
    pred = transcribe(c["audio"])
    print(f"  ref: {c['transcript'][:50]}")
    print(f"  hyp: {pred[:50]}")
    print(f"  WER: {jiwer_wer(c['transcript'].lower(), pred.lower()):.2f}\n")



Loading 50 clips from LibriSpeech test-clean …


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Scanning: 134it [00:05, 24.48it/s]


✓ Loaded 50 clips.

Baseline WER on clean audio:
  ref: CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS
  hyp: CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS
  WER: 0.00

  ref: YOU WILL BE FRANK WITH ME I ALWAYS AM
  hyp: YOU WILL BE FRANK WITH ME I ALWAYS AM
  WER: 0.00

  ref: CAN YOU IMAGINE WHY BUCKINGHAM HAS BEEN SO VIOLENT
  hyp: CAN YOU IMAGINE WHY BUCKINGHAM HAS BEEN SO VIOLENT
  WER: 0.00



In [ ]:
# @title Standard C&W Attack

def run_cw(audio, target, c=50.0, lr=0.001, iters=None, label="C&W"):
    """Standard Carlini & Wagner targeted attack"""

    orig_t = torch.tensor(audio, dtype=torch.float32, device=DEVICE)
    x_clip = torch.clamp(orig_t, -0.9999, 0.9999)
    w      = torch.arctanh(x_clip).clone().detach().requires_grad_(True)

    optim   = torch.optim.Adam([w], lr=lr)
    history = []
    success_step = None

    for step in tqdm(range(iters), desc=f"  {label}"):

        optim.zero_grad()

        x_adv = torch.tanh(w).unsqueeze(0)

        ctc_l = ctc_loss_fn(x_adv, target)
        delta = x_adv.squeeze(0) - orig_t
        l2_l  = torch.sum(delta ** 2)

        loss = l2_l + c * ctc_l
        loss.backward()
        optim.step()

        history.append((ctc_l.item(), l2_l.item()))

        # Check success every 5 Steps
        adv_np = torch.tanh(w).detach().cpu().numpy()
        if step % 5 == 0:
            pred = transcribe(adv_np)

        if pred.upper().strip() == target.upper().strip():
            success_step = step + 1
            tqdm.write(f"    ✓ Success at step {success_step}")
            break

        # Print progress every 200 steps
        if (step + 1) % 200 == 0:
            tqdm.write(
                f"    Step {step+1}: "
                f"CTC={ctc_l.item():.4f} | "
                f"L2={l2_l.item():.2f} | "
                f"'{pred}'"
            )

    # Final adversarial example
    adv_np = torch.tanh(w).detach().cpu().numpy().astype(np.float32)
    final_transcript = transcribe(adv_np)

    return {
        "adv": adv_np,
        "pert": adv_np - audio,
        "transcript": final_transcript,
        "success": final_transcript.upper().strip() == target.upper().strip(),
        "success_step": success_step,
        "l2": float(np.linalg.norm(adv_np - audio)),
        "history": history,
    }

In [ ]:
# @title Psychoacoustic C&W Attack

def run_psycho_cw(audio, target, c=60.0, lr=0.001, iters=None):
    """C&W with spectral masking + L2 regularization"""

    orig_t = torch.tensor(audio, dtype=torch.float32, device=DEVICE)
    x_clip = torch.clamp(orig_t, -0.9999, 0.9999)
    w      = torch.arctanh(x_clip).clone().detach().requires_grad_(True)

    optim   = torch.optim.Adam([w], lr=lr)
    history = []
    success_step = None

    # Precompute original STFT magnitude
    with torch.no_grad():
        orig_stft = torch.stft(orig_t, n_fft=512, return_complex=True)
        orig_mag  = torch.abs(orig_stft)

    for step in tqdm(range(iters), desc="  PsychoC&W"):

        optim.zero_grad()

        x_adv = torch.tanh(w).unsqueeze(0)
        delta = x_adv.squeeze(0) - orig_t

        # CTC loss
        ctc_l = ctc_loss_fn(x_adv, target)

        # L2 loss
        l2_l = torch.sum(delta ** 2)

        # Psychoacoustic penalty
        delta_stft = torch.stft(delta, n_fft=512, return_complex=True)
        delta_mag  = torch.abs(delta_stft)

        eps   = 1e-6
        ratio = delta_mag / (orig_mag + eps)

        # Allow up to 30% relative increase before penalizing
        excess   = torch.relu(ratio - 0.30)
        psycho_l = torch.mean(excess ** 2)

        alpha = 80.0  # psycho weight

        # Final loss
        loss = l2_l + c * ctc_l + alpha * psycho_l

        loss.backward()
        optim.step()

        history.append((ctc_l.item(), l2_l.item()))

        # Check success EVERY 5 step
        adv_np = torch.tanh(w).detach().cpu().numpy()
        if step % 5 == 0:
            pred = transcribe(adv_np)

        if pred.upper().strip() == target.upper().strip():
            success_step = step + 1
            tqdm.write(f"    ✓ Success at step {success_step}")
            break

        # Print progress every 200 steps
        if (step + 1) % 200 == 0:
            tqdm.write(
                f"    Step {step+1}: "
                f"CTC={ctc_l.item():.4f} | "
                f"L2={l2_l.item():.2f} | "
                f"Psy={psycho_l.item():.4f} | "
                f"'{pred}'"
            )

    adv_np = torch.tanh(w).detach().cpu().numpy().astype(np.float32)
    final_transcript = transcribe(adv_np)

    return {
        "adv": adv_np,
        "pert": adv_np - audio,
        "transcript": final_transcript,
        "success": final_transcript.upper().strip() == target.upper().strip(),
        "success_step": success_step,
        "l2": float(np.linalg.norm(adv_np - audio)),
        "history": history,
    }

In [ ]:
# @title Run Full Experiment

ITERS = 2000

all_results = []
for i, clip in enumerate(clips):
    print(f"\n{'='*60}")
    print(f"Clip {i+1}/{len(clips)}: {clip['id']}")
    print(f"Original transcript: {clip['transcript'][:60]}")
    print(f"Target: '{TARGET_TEXT}'")

    std_r   = run_cw(clip["audio"], TARGET_TEXT, iters=ITERS, label="Standard C&W")
    psych_r = run_psycho_cw(clip["audio"], TARGET_TEXT, iters=ITERS)

    std_snr   = snr_db(clip["audio"], std_r["pert"])
    psych_snr = snr_db(clip["audio"], psych_r["pert"])

    print(f"\n  STANDARD: success={std_r['success']}  SNR={std_snr:.1f}dB  L2={std_r['l2']:.4f}  → '{std_r['transcript']}'")
    print(f"  PSYCHO:   success={psych_r['success']}  SNR={psych_snr:.1f}dB  L2={psych_r['l2']:.4f}  → '{psych_r['transcript']}'")

    # Save audio with index prefix (1-based, zero-padded)
    index_str = f"{i+1:03d}"

    sf.write(f"results/audio/{index_str}_{clip['id']}_orig.wav",
            clip["audio"], SAMPLE_RATE)

    sf.write(f"results/audio/{index_str}_{clip['id']}_std_adv.wav",
            std_r["adv"], SAMPLE_RATE)

    sf.write(f"results/audio/{index_str}_{clip['id']}_psych_adv.wav",
            psych_r["adv"], SAMPLE_RATE)

    all_results.append({
        "index":               i + 1,   # ← 1-based index
        "clip_id":             clip["id"],
        "original_transcript": clip["transcript"],
        "target_text":         TARGET_TEXT,
        "std_success":         int(std_r["success"]),
        "std_success_step": std_r["success_step"],
        "std_l2":              std_r["l2"],
        "std_snr_db":          std_snr,
        "std_transcript":      std_r["transcript"],
        "psych_success":       int(psych_r["success"]),
        "psych_success_step": psych_r["success_step"],
        "psych_l2":            psych_r["l2"],
        "psych_snr_db":        psych_snr,
        "psych_transcript":    psych_r["transcript"],
        "snr_gain": psych_snr - std_snr,
    })

df = pd.DataFrame(all_results)
df.set_index("index", inplace=True)
df.to_csv("results/results.csv")
print("\n✓ Results saved to results/results.csv")
# print(df[["clip_id","std_success","std_snr_db","psych_success","psych_snr_db"]].to_string())
print(df[["clip_id","std_success","std_snr_db","psych_success","psych_snr_db"]])




Clip 1/50: 6930-75918-0000
Original transcript: CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS
Target: 'OPEN THE FRONT DOOR'


  Standard C&W:  10%|█         | 201/2000 [00:30<03:13,  9.30it/s]

    Step 200: CTC=0.5175 | L2=6.02 | 'OPEN THE FNT DOOR'


  Standard C&W:  12%|█▏        | 240/2000 [00:35<04:20,  6.76it/s]


    ✓ Success at step 241


...

    Step 600: CTC=1.0462 | L2=4.36 | Psy=0.4445 | 'PETI RONT DO'


  PsychoC&W:  40%|████      | 801/2000 [01:36<02:21,  8.45it/s]

    Step 800: CTC=0.6148 | L2=4.51 | Psy=0.4696 | 'OPEN SH RONT DOO'


  PsychoC&W:  44%|████▍     | 875/2000 [01:45<02:15,  8.28it/s]


    ✓ Success at step 876

  STANDARD: success=True  SNR=24.5dB  L2=2.5612  → 'OPEN THE FRONT DOOR'
  PSYCHO:   success=True  SNR=26.0dB  L2=2.1478  → 'OPEN THE FRONT DOOR'

✓ Results saved to results/results.csv
                clip_id  std_success  std_snr_db  psych_success  psych_snr_db
index                                                                        
1       6930-75918-0000            1   19.669500              1     20.325489
2       6930-75918-0007            1   17.883348              1     21.520378
3       6930-75918-0008            1   18.996143              1     22.528576
4       6930-75918-0010            1   20.923071              1     20.967636
5       6930-75918-0011            1   21.232822              1     22.158504
6       6930-75918-0013            1   18.115513              1     21.295341
7       6930-76324-0000            1   19.723881              1     20.410236
8       6930-76324-0001            1   18.862671              1     23.804691
9      

In [ ]:
# @title Print Summary Statistics

n = len(df)
print("EXPERIMENT SUMMARY:")
print(f"\nClips evaluated       : {n}")
print(f"\nStandard C&W Attack")
print(f"  Success Rate        : {df['std_success'].sum()}/{n} ({100*df['std_success'].mean():.1f}%)")
print(f"  Mean SNR (dB)       : {df['std_snr_db'].mean():.2f}")
print(f"  Mean L2 norm        : {df['std_l2'].mean():.4f}")
print(f"  Mean success step: {df['std_success_step'].mean():.1f}")
print(f"\nPsychoacoustic C&W Attack")
print(f"  Success Rate        : {df['psych_success'].sum()}/{n} ({100*df['psych_success'].mean():.1f}%)")
print(f"  Mean SNR (dB)       : {df['psych_snr_db'].mean():.2f}")
print(f"  Mean L2 norm        : {df['psych_l2'].mean():.4f}")
print(f"  Mean success step:   {df['psych_success_step'].mean():.1f}")
snr_gain = df['psych_snr_db'].mean() - df['std_snr_db'].mean()
print(f"\nPsychoacoustic attack improves SNR by {snr_gain:+.2f} dB")
print("  (Positive = psychoacoustic attack is stealthier/less audible)")

from scipy import stats
# Paired t-test on per-clip SNR values
t_stat, p_val = stats.ttest_rel(df["psych_snr_db"], df["std_snr_db"])

print(f"\nPaired t-test on SNR:")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value:     {p_val:.6f}")

# Cohen's d
snr_diff = df["psych_snr_db"] - df["std_snr_db"]
cohens_d = snr_diff.mean() / snr_diff.std(ddof=1)
print(f"Cohen's d:   {cohens_d:.4f}")


EXPERIMENT SUMMARY:

Clips evaluated       : 50

Standard C&W Attack
  Success Rate        : 50/50 (100.0%)
  Mean SNR (dB)       : 19.23
  Mean L2 norm        : 2.4229
  Mean success step: 322.1

Psychoacoustic C&W Attack
  Success Rate        : 50/50 (100.0%)
  Mean SNR (dB)       : 21.63
  Mean L2 norm        : 1.8558
  Mean success step:   721.3

Psychoacoustic attack improves SNR by +2.40 dB
  (Positive = psychoacoustic attack is stealthier/less audible)

Paired t-test on SNR:
T-statistic: 11.9155
P-value:     0.000000
Cohen's d:   1.6851


In [ ]:
# @title Generate All Figures

import os
os.makedirs("results/figures", exist_ok=True)

n = len(df)

# Figure 1: SNR Comparison (All Clips)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(1, n + 1)

ax.bar(x - 0.2, df["std_snr_db"],   0.4, label="Standard C&W",      color="steelblue")
ax.bar(x + 0.2, df["psych_snr_db"], 0.4, label="Psychoacoustic C&W", color="darkorange")

ax.set_xlabel("Audio Clip Index")
ax.set_ylabel("SNR (dB) — Higher = More Stealthy")
ax.set_title("SNR Comparison: Standard vs Spectral-Masked C&W")
ax.set_xlim(0.5, n + 0.5)
ax.set_xticks(range(1, n + 1, 5))
ax.legend()

plt.tight_layout()
plt.savefig("results/figures/snr_comparison.png", dpi=200)
plt.show()

print("✓ Figure 1 saved: SNR comparison")


# Figure 2: Spectrogram Comparison (First Clip)

c0 = clips[0]
audio = c0["audio"]
index_str = f"{1:03d}"

std_adv_audio, _   = sf.read(f"results/audio/{index_str}_{c0['id']}_std_adv.wav")
psych_adv_audio, _ = sf.read(f"results/audio/{index_str}_{c0['id']}_psych_adv.wav")

fig, axes = plt.subplots(3, 1, figsize=(12, 9))

for ax, sig, title in zip(
    axes,
    [audio, std_adv_audio, psych_adv_audio],
    ["Original", "Standard C&W Adversarial", "Psychoacoustic C&W Adversarial"],
):
    D = librosa.amplitude_to_db(
        np.abs(librosa.stft(sig, n_fft=2048)), ref=np.max
    )
    librosa.display.specshow(
        D, sr=SAMPLE_RATE, x_axis="time", y_axis="hz", ax=ax, cmap="magma"
    )
    ax.set_title(title)

plt.suptitle(f"Spectrogram Comparison — {c0['id']}")
plt.tight_layout()
plt.savefig("results/figures/spectrogram_comparison.png", dpi=200)
plt.show()

print("✓ Figure 2 saved: Spectrogram comparison")


# Figure 3: Perturbation vs Masking Threshold (First Clip)

freqs = librosa.fft_frequencies(sr=SAMPLE_RATE, n_fft=2048)
freqs[0] = 1e-6   # avoid zero frequency
thresh = masking_threshold(audio)
thresh_avg = np.mean(thresh, axis=1)

std_pert   = std_adv_audio - audio
psych_pert = psych_adv_audio - audio

def avg_power(sig):
    return np.mean(np.abs(librosa.stft(sig, n_fft=2048))**2, axis=1)

fig, ax = plt.subplots(figsize=(12, 5))

ax.semilogy(freqs, avg_power(audio),
            color="steelblue", label="Original", linewidth=1.5, alpha=0.7)

ax.semilogy(freqs, thresh_avg,
            color="green", linestyle="--", linewidth=2, label="Masking Threshold")

ax.semilogy(freqs, avg_power(std_pert),
            color="red", linewidth=1.5, label="Standard Perturbation")

ax.semilogy(freqs, avg_power(psych_pert),
            color="darkorange", linewidth=1.5, label="Psycho Perturbation")

ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power (log scale)")
ax.set_title("Perturbation Power vs Masking Threshold")
ax.set_xlim(0, 8000)
ax.legend()

plt.tight_layout()
plt.savefig("results/figures/perturbation_spectrum.png", dpi=200)
plt.show()

print("✓ Figure 3 saved: Perturbation spectrum")



✓ Figure 1 saved: SNR comparison
✓ Figure 2 saved: Spectrogram comparison
✓ Figure 3 saved: Perturbation spectrum


In [ ]:
# @title Download All Results to Computer

import shutil
from google.colab import files

# Zip up all results
shutil.make_archive("adversarial_results", "zip", "results")
files.download("adversarial_results.zip")
print("✓ Download started! Check your Downloads folder.")
print("  Contents of the ZIP:")
print("    results/results.csv          ← all metrics")
print("    results/figures/*.png        ← all figures")
print("    results/audio/*.wav          ← original + adversarial audio files")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download started! Check your Downloads folder.
  Contents of the ZIP:
    results/results.csv          ← all metrics
    results/figures/*.png        ← all figures
    results/audio/*.wav          ← original + adversarial audio files
